# StatsBomb xG Data Preparation
Load a reproducible sample of 500 open-data matches, keep non-penalty shots, and retain only the raw fields needed for the project.

In [ ]:
!pip -q install statsbombpy tqdm

import pandas as pd
from statsbombpy import sb
from tqdm.auto import tqdm

TARGET_MATCHES = 500
RANDOM_STATE = 42

RAW_INPUT_FIELDS = [
    "location",
    "shot_body_part",
    "shot_technique",
    "shot_first_time",
    "shot_type",
    "play_pattern",
    "under_pressure",
]

KEEP_COLUMNS = [
    "match_id",
    "id",
    "team",
    "player",
    *RAW_INPUT_FIELDS,
    "shot_outcome",
    "shot_statsbomb_xg",
]


## 1. Get available matches

In [ ]:
competitions = sb.competitions()

match_frames = []

competition_seasons = competitions[
    ["competition_id", "season_id"]
].drop_duplicates()

for row in tqdm(
    competition_seasons.itertuples(index=False),
    total=len(competition_seasons),
    desc="Loading match lists"
):
    try:
        matches = sb.matches(
            competition_id=int(row.competition_id),
            season_id=int(row.season_id)
        )

        match_columns = [
            c for c in
            ["match_id", "match_date", "competition", "season", "home_team", "away_team"]
            if c in matches.columns
        ]

        match_frames.append(matches[match_columns])

    except Exception as e:
        print(
            f"Skipped competition {row.competition_id}, "
            f"season {row.season_id}: {e}"
        )

all_matches = (
    pd.concat(match_frames, ignore_index=True)
      .drop_duplicates(subset="match_id")
      .reset_index(drop=True)
)

candidate_matches = (
    all_matches.sample(frac=1, random_state=RANDOM_STATE)
               .reset_index(drop=True)
)

print(f"Available open-data matches: {len(all_matches):,}")
print(f"Target sample: {min(TARGET_MATCHES, len(all_matches)):,} matches")


## 2. Extract non-penalty shots

In [ ]:
shot_frames = []
successful_match_ids = []
failed_match_ids = []

for match_id in tqdm(candidate_matches["match_id"], desc="Loading events"):
    if len(successful_match_ids) >= TARGET_MATCHES:
        break

    try:
        events = sb.events(match_id=int(match_id))

        match_shots = events.loc[events["type"].eq("Shot")].copy()
        match_shots = match_shots.loc[
            ~match_shots["shot_type"].eq("Penalty")
        ].copy()

        # Keep the selected raw fields only.
        for column in KEEP_COLUMNS:
            if column not in match_shots.columns:
                match_shots[column] = pd.NA

        shot_frames.append(match_shots[KEEP_COLUMNS])
        successful_match_ids.append(int(match_id))

    except Exception as e:
        failed_match_ids.append(int(match_id))
        print(f"Skipped match {match_id}: {e}")

shots = pd.concat(shot_frames, ignore_index=True)

used_matches = all_matches[
    all_matches["match_id"].isin(successful_match_ids)
].copy()

print(f"Matches loaded: {len(successful_match_ids):,}")
print(f"Non-penalty shots: {len(shots):,}")
print(f"Saved columns: {len(shots.columns)}")
print(f"Failed matches: {len(failed_match_ids)}")

shots.head()


## 3. Dataset summary

In [ ]:
goal_count = shots["shot_outcome"].eq("Goal").sum()
non_goal_count = len(shots) - goal_count
goal_rate = goal_count / len(shots) if len(shots) else 0

summary = pd.DataFrame({
    "Statistic": [
        "Matches",
        "Non-penalty shots",
        "Raw model input fields",
        "Total columns in saved shot file",
        "Target classes",
        "Goals",
        "Non-goals",
        "Goal rate",
    ],
    "Value": [
        len(successful_match_ids),
        len(shots),
        len(RAW_INPUT_FIELDS),
        len(shots.columns),
        2,
        int(goal_count),
        int(non_goal_count),
        f"{goal_rate:.2%}",
    ]
})

display(summary)


In [ ]:
feature_summary = pd.DataFrame({
    "Raw field": RAW_INPUT_FIELDS,
    "Role": [
        "Shot location coordinates",
        "Body part used for the shot",
        "Shot technique",
        "Whether the shot was first-time",
        "Shot type",
        "Play pattern leading to the shot",
        "Whether the shooter was under pressure",
    ]
})

display(feature_summary)


## 4. Representative examples

In [ ]:
goal_examples = shots[shots["shot_outcome"].eq("Goal")].sample(
    n=min(3, goal_count),
    random_state=RANDOM_STATE
)

non_goal_examples = shots[~shots["shot_outcome"].eq("Goal")].sample(
    n=min(3, non_goal_count),
    random_state=RANDOM_STATE
)

example_rows = pd.concat(
    [goal_examples, non_goal_examples],
    ignore_index=True
)

display(
    example_rows[
        [
            "location",
            "shot_body_part",
            "shot_technique",
            "shot_first_time",
            "shot_type",
            "play_pattern",
            "under_pressure",
            "shot_outcome",
        ]
    ]
)


## 5. Export

In [ ]:
shots.to_csv("statsbomb_non_penalty_shots_raw.csv", index=False)
used_matches.to_csv("statsbomb_selected_matches.csv", index=False)
summary.to_csv("statsbomb_dataset_summary.csv", index=False)

print("Saved:")
print("- statsbomb_non_penalty_shots_raw.csv")
print("- statsbomb_selected_matches.csv")
print("- statsbomb_dataset_summary.csv")


**Note:** `shot_statsbomb_xg` is retained only as a later benchmark. It must not be used as a model input. Distance, angle, encoded categories, and the binary goal target are intentionally not created in this preparation notebook.